In [1]:
import numpy as np
import pandas as pd
import os
import joblib
import pickle
import math
import ast

from scipy.stats import median_abs_deviation, hypergeom, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

%matplotlib inline

figure_params = {
    'dpi': 300, 
    'bbox_inches': 'tight', 
    'format': 'svg', 
    'transparent': True
}

plt.rcParams.update({
    "text.usetex": False,
    "svg.fonttype": 'none'
})

# Saving plots with editable text
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts (editable text)

In [2]:
import sys
# Ensure this analysis directory is importable regardless of kernel CWD
_here = '/projects/bhdw/asachan/methods/FIREFate/multiome_dynamic_regulation/py_scripts/analysis'
if _here not in sys.path:
    sys.path.insert(0, _here)

import dictys
from utils_custom import *
from pseudotime_curves import *
from episodic_dynamics import *
from config import *

In [3]:
import importlib
import firefate.utils.plots, firefate.utils.custom
import state_dynamics
# re-bind
importlib.reload(firefate.utils.plots)
importlib.reload(firefate.utils.custom)
importlib.reload(state_dynamics)
from state_dynamics import TFForceWaves, StateFrequency
from dynamic_validation import TFForceValidation

In [4]:
config = Config()

In [5]:
# Load data
dictys_dynamic_object = dictys.net.dynamic_network.from_file('/work/nvme/bhdw/asachan/data_files/firefate/bcell/outs/dynamic.h5')

### Defining lineage trajectories

In [ ]:
PB_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(98, 147, 1)) + [2]
GC_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(147, 193, 1)) + [3]
PB_post_bifurcation_window_indices = [0] + list(range(98, 147, 1)) + [2]
GC_post_bifurcation_window_indices = [0] + list(range(147, 193, 1)) + [3]

In [ ]:
# Define distinct colors for better visibility
colors_cell_count = {
    'ActB-1': '#87CEFA',     # lightskyblue
    'ActB-2': '#1E90FF',     # dodgerblue
    'ActB-3': '#00008B',     # darkblue
    'ActB-4': '#9370DB',     # mediumorchid
    'GC-1': '#7BDE7B',       # custom light green
    'GC-2': '#008000',       # green
    'PB-2': '#BB3636',       # custom red
    'earlyActB': '#008080',   # teal
    'earlyPB': '#F08080'   # lightcoral
}

### Loading prioritized tfs

In [ ]:
ss_firefate_combined = pd.read_csv('/projects/bhdw/asachan/tmp/ss_firefate_links_2B.csv')

In [ ]:
display(ss_firefate_combined)

In [ ]:
# Build (TF, target) tuples, keeping only links whose TF and target are both
# present in the dictys object. get_beta_curves can only compute forces for genes
# in the GRN (TFs in nids[0], targets in ndict); filtering here makes the computed
# link set explicit rather than relying on the internal skip.
all_links = list(zip(ss_firefate_combined['source'], ss_firefate_combined['target']))

_, _, missing_tfs = get_tf_indices(dictys_dynamic_object, list({tf for tf, _ in all_links}))
missing_tfs = set(missing_tfs)
ndict = dictys_dynamic_object.ndict

ss_firefate_combined_tuple = [(tf, tg) for tf, tg in all_links
                     if tf not in missing_tfs and tg in ndict]
dropped = [(tf, tg) for tf, tg in all_links
           if tf in missing_tfs or tg not in ndict]
print(f"{len(ss_firefate_combined_tuple)}/{len(all_links)} links kept; "
      f"{len(dropped)} dropped (TF/target absent from GRN): {dropped}")

In [ ]:
# load the episodically enriched links from file
episodic_links_file = '/projects/bhdw/asachan/papers/firefate/figures/enriched_tf_lf_targets_per_episode.csv'
episodic_links = pd.read_csv(episodic_links_file)

In [ ]:
display(episodic_links)

In [ ]:
#create tuples from the TF and genes_in_lf list of comma separated targets and take union of all links across all rows to make a unique set of links
episodic_links_tuples = set()
for _, row in episodic_links.iterrows():
    tf = row['TF']
    targets = row['genes_in_lf'].split(',')
    for target in targets:
        episodic_links_tuples.add((tf, target.strip()))


In [ ]:
display(len(episodic_links_tuples))

# TF forces

In [ ]:
# TF forces over pseudotime (expression / regulation curves cached internally)
waves_pb = TFForceWaves(
    dictys_dynamic_object,
    trajectory_range=(0, 2),
    num_points=100,
    dist=0.0005,
    sparsity=0.01,
)

In [ ]:
# GC branch: same TF-forces machinery over the GC trajectory (node 0 -> 3).
waves_gc = TFForceWaves(
    dictys_dynamic_object,
    trajectory_range=(0, 3),
    num_points=100,
    dist=0.0005,
    sparsity=0.01,
)

### Cross-branch pooled comparison (PB ∪ GC)

In [ ]:
# Combined 3-box comparison: state-specific enriched, episodic enriched, and ONE shared
# random null (size-matched to the larger of the two enriched sets; drawn with every
# TF/target from BOTH sets removed). Each link scored by abs-max TF force across PB & GC.
combo_df = TFForceValidation.compare_sets(
    {'PB': waves_pb, 'GC': waves_gc},
    {'State-specific': ss_firefate_combined_tuple,
     'Episodic': episodic_links_tuples},
    varname='w_in', exclude='tf_and_target', random_state=0,
)
display(combo_df['group'].value_counts())

fig, ax = TFForceValidation.plot_multi(
    combo_df,
    group_order=['State-specific', 'Episodic', 'random'],
    group_labels=['State-specific\nenriched', 'Episodic\nenriched', 'Random\n(non-enriched)'],
    ylabel='abs(max TF-force)',
)
# fig.savefig('/projects/bhdw/asachan/papers/firefate/figures/combined_links_validation.svg',
#             format='svg', bbox_inches='tight')
plt.show()

## Phase definitions

In [ ]:
# Class 1: cell-state fate frequencies + extrema / termination pseudotimes
sf_pb = StateFrequency(
    dictys_dynamic_object,
    cell_labels=config.CELL_LABELS,
    colors=colors_cell_count,
    trajectory_range=(0, 2),
)
pseudotime_values_of_windows_pb = sf_pb.pseudotime_values_of_windows

In [ ]:
# Class 1: cell-state fate frequencies + extrema / termination pseudotimes
sf_gc = StateFrequency(
    dictys_dynamic_object,
    cell_labels=config.CELL_LABELS,
    colors=colors_cell_count,
    trajectory_range=(0, 3),
)
pseudotime_values_of_windows_gc = sf_gc.pseudotime_values_of_windows

In [ ]:
# PB switch boundaries = termination pseudotimes of ActB-4 then earlyPB (3 phases).
pb_switches = [sf_pb.termination_pseudotime(s, PB_post_bifurcation_window_indices,
                                            method='threshold')
               for s in ['ActB-4', 'earlyPB']]

# GC switch boundary = termination pseudotime of ActB-3 (2 phases).
gc_switches = [sf_gc.termination_pseudotime('ActB-3', GC_post_bifurcation_window_indices,
                                            method='threshold')]

### TF Force Phase split validation

In [ ]:
import importlib, dynamic_validation
importlib.reload(dynamic_validation)
from dynamic_validation import TFForceValidation

# PB switch boundaries = termination pseudotimes of ActB-4 then earlyPB (3 phases).
pb_switches = [sf_pb.termination_pseudotime(s, PB_post_bifurcation_window_indices,
                                            method='threshold')
               for s in ['ActB-4', 'earlyPB']]

# Classify the universe of enriched links (state-specific + episodic) into PB phases,
# keeping only links whose winning branch is PB, then compare the two enriched sets
# against ONE shared per-phase random null. Each point is a link's abs-max TF force,
# which (since phase = where the force peaks) is its in-phase peak force.
pb_phase_df = TFForceValidation.compare_sets_by_phase(
    {'PB': waves_pb, 'GC': waves_gc},
    {'State-specific': ss_firefate_combined_tuple,
     'Episodic': episodic_links_tuples},
    switch_pseudotimes={'PB': pb_switches},
    varname='w_in', exclude='tf_and_target', random_state=0,
)
display(pb_phase_df.groupby(['phase', 'group']).size().unstack(fill_value=0))

In [ ]:
# GC switch boundary = termination pseudotime of ActB-3 (2 phases).
gc_switches = [sf_gc.termination_pseudotime('ActB-3', GC_post_bifurcation_window_indices,
                                            method='threshold')]

# Classify the universe of enriched links (state-specific + episodic) into GC phases,
# keeping only links whose winning branch is GC, then compare the two enriched sets
# against ONE shared per-phase random null. Each point is a link's abs-max TF force,
# which (since phase = where the force peaks) is its in-phase peak force.
gc_phase_df = TFForceValidation.compare_sets_by_phase(
    {'PB': waves_pb, 'GC': waves_gc},
    {'State-specific': ss_firefate_combined_tuple,
     'Episodic': episodic_links_tuples},
    switch_pseudotimes={'GC': gc_switches},
    varname='w_in', exclude='tf_and_target', random_state=0,
)
display(gc_phase_df.groupby(['phase', 'group']).size().unstack(fill_value=0))

In [ ]:
fig, ax = TFForceValidation.plot_multi_by_phase(
    pb_phase_df, branch='PB',
    group_order=['State-specific', 'Episodic', 'random'],
    group_labels=['State-specific', 'Episodic', 'Random (non-enriched)'],
    ylabel='Abs max TF force (in-phase)',
    xlabel= ['Phase 1', 'Phase 2', 'Phase 3']
)
ax.set_title('PB phases')
fig.savefig('/projects/bhdw/asachan/papers/firefate/figures/pb_phase_links_validation.svg',
            format='svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = TFForceValidation.plot_multi_by_phase(
    gc_phase_df, branch='GC',
    group_order=['State-specific', 'Episodic', 'random'],
    group_labels=['State-specific', 'Episodic', 'Random (non-enriched)'],
    ylabel='Abs max TF force (in-phase)',
    xlabel=['Phase 1', 'Phase 2'],
)
ax.set_title('GC phases')
fig.savefig('/projects/bhdw/asachan/papers/firefate/figures/gc_phase_links_validation.svg',
            format='svg', bbox_inches='tight')
plt.show()

# TF Binding Dynamics

In [ ]:
base_path = '/work/hdd/bgdb/asachan/datasets_hdd/B_cell_dictys_actb1_added_v2/dictys_related/tmp_dynamic/'
from firefate.core.pseudotime_curves import SmoothedCurvesChromatin

smooth_chromatin_object = SmoothedCurvesChromatin(
    tfs=None,
    base_path=base_path
)


In [ ]:
# Get pseudotime values for all windows
aligner_pb = AlignTimeScales(
    dictys_dynamic_object=dictys_dynamic_object,
    trajectory_range=(1, 2),  # any trajectory starting from 1
    num_points=40,
    dist=0.001,
    sparsity=0.01
)
aligner_gc = AlignTimeScales(
    dictys_dynamic_object=dictys_dynamic_object,
    trajectory_range=(1, 3),  # any trajectory starting from 1
    num_points=40,
    dist=0.001,
    sparsity=0.01
)
window_pseudotimes = aligner_pb.pseudotime_of_windows()

In [ ]:
smooth_chromatin_object.set_trajectory_info(
    pb_indices=[1] + list(range(97, 3, -1)) + [0] + list(range(98, 147, 1)) + [2],
    gc_indices=[1] + list(range(97, 3, -1)) + [0] + list(range(147, 193, 1)) + [3],
    window_pseudotimes=window_pseudotimes
)

In [21]:
smooth_chromatin_object.extract_data(n_windows=194, n_processes=None)
smooth_chromatin_object.process_dynamics()   # populates series_pb / series_gc for BindingPhases

Extracting Binding Data:   0%|                                                                                                                                    | 0/194 [04:52<?, ?it/s]Process ForkPoolWorker-67:
Process ForkPoolWorker-52:
Process ForkPoolWorker-51:
Process ForkPoolWorker-69:
Process ForkPoolWorker-94:
Process ForkPoolWorker-85:
Process ForkPoolWorker-106:
Process ForkPoolWorker-77:
Process ForkPoolWorker-49:
Process ForkPoolWorker-65:
Process ForkPoolWorker-80:
Process ForkPoolWorker-124:
Process ForkPoolWorker-47:
Process ForkPoolWorker-64:
Process ForkPoolWorker-118:
Process ForkPoolWorker-12:
Process ForkPoolWorker-107:
Process ForkPoolWorker-45:
Process ForkPoolWorker-32:
Process ForkPoolWorker-14:
Process ForkPoolWorker-71:
Process ForkPoolWorker-89:
Process ForkPoolWorker-50:
Process ForkPoolWorker-117:
Process ForkPoolWorker-39:
Process ForkPoolWorker-13:
Process ForkPoolWorker-60:
Process ForkPoolWorker-113:
Process ForkPoolWorker-99:
Process ForkPoolWorker-74

KeyboardInterrupt: 

In [ ]:
from state_dynamics import BindingPhases

category_tfs = {
    'Static':   BindingPhases.tfs_from_links(ss_firefate_combined_tuple),
    'Episodic': BindingPhases.tfs_from_links(episodic_links_tuples),
}

# --- per-phase BOX plots (the per-phase top-5 selection) ---
bp_pb = BindingPhases(pb_switches, smooth_chromatin_object, 'pb')   # 3 phases
bp_gc = BindingPhases(gc_switches, smooth_chromatin_object, 'gc')   # 2 phases

display(bp_pb.top_tfs_table(category_tfs, top_k=5))     # which TFs top each phase
fig, ax = bp_pb.plot_box(category_tfs, top_k=5, annotate=True); fig.show()
fig, ax = bp_gc.plot_box(category_tfs, top_k=5, annotate=True); fig.show()

# --- continuous curves: a SEPARATE concern, from the pseudotime_curves module ---
# (binding score / OCR over pseudotime — uses SmoothedCurvesChromatin directly,
#  not the per-phase selection)
# smooth_chromatin_object.plot(some_categories_dict, ...)
# smooth_chromatin_object.plot_score_vs_count_comparison(some_categories_dict, ...)